# Badminton Shot Classification - Deep Learning Training

**Phase 1.5: ROI-based Single Player Pose Extraction**

This notebook trains deep learning models for badminton shot classification using skeleton-based action recognition.

## Models Implemented

1. **ST-GCN** (Spatial Temporal Graph Convolutional Network) - Graph-based baseline
2. **MS-G3D** (Multi-Scale Graph 3D) - State-of-the-art GCN with multi-scale aggregation
3. **BiLSTM** (Bidirectional LSTM) - Temporal baseline for comparison
4. **Transformer** (Skeleton Transformer) - Attention-based model

## Research Background

Based on recent research (2024-2025):
- [Deep learning-based badminton action recognition](https://journals.sagepub.com/doi/10.1177/1088467X251353444) - SlowFast + Siamese: 83.08% Top-1 accuracy
- [Strategy analysis using deep learning](https://www.sciencedirect.com/science/article/abs/pii/S2542660524002014) - 2D-CNN + LSTM: 90.9% accuracy
- [ST-GCN for skeleton-based action recognition](https://arxiv.org/abs/1801.07455) - First GCN + action recognition
- [MS-G3D: Disentangling Graph Convolutions](https://arxiv.org/abs/2003.14111) - CVPR 2020, state-of-the-art
- [Two-stream GCN-Transformer](https://www.nature.com/articles/s41598-025-87752-8) - 2025, combining GCN + Transformer

---

## 1. Setup and Installation

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️  WARNING: CUDA not available. Training will be slow on CPU.")

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn matplotlib seaborn pandas
!pip install -q tqdm tensorboard

print("✓ Packages installed")

In [ ]:
# Authenticate to GCS (if running in Colab)
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✓ Authenticated to Google Cloud")
except:
    print("Not running in Colab or already authenticated")

## 2. Download Data from GCS

In [ ]:
# Download poses and metadata from GCS
import os
from pathlib import Path

# Create data directories
!mkdir -p data/poses

print("Downloading metadata...")
!gsutil cp gs://iti123storage/data/metadata_roi.csv data/metadata.csv

print("\nDownloading poses (this may take 5-10 minutes)...")
!gsutil -m rsync -r gs://iti123storage/features/poses_roi/ data/poses/

# Verify download
pose_count = len(list(Path('data/poses').glob('*.pkl')))
print(f"\n✓ Downloaded {pose_count} pose files")

## 3. Data Loading and Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from collections import Counter

# Load metadata
metadata = pd.read_csv('data/metadata.csv')

print(f"Total clips: {len(metadata)}")
print(f"\nShot type distribution:")
print(metadata['shot_type'].value_counts())

# Check class balance
class_counts = metadata['shot_type'].value_counts()
print(f"\nClass balance:")
for shot, count in class_counts.items():
    print(f"  {shot}: {count} ({count/len(metadata)*100:.1f}%)")

In [ ]:
# Filter: Load only clips with existing pose files
poses_dir = Path('data/poses')
pose_files = {f.stem for f in poses_dir.glob('*.pkl')}

# Filter metadata to only include clips with poses
metadata = metadata[metadata['video_id'].isin(pose_files)].reset_index(drop=True)

print(f"Clips with poses: {len(metadata)}")
print(f"Success rate: {len(metadata) / len(pose_files) * 100:.1f}%")

In [ ]:
# Quality filters
MIN_FRAMES = 30  # Minimum sequence length (1 second at 30 FPS)
MAX_FRAMES = 300  # Maximum sequence length (10 seconds)
MULTI_PLAYER_THRESHOLD = 0.6  # X-range threshold for multi-player detection

def load_and_filter_pose(video_id):
    """Load pose and check quality filters"""
    pose_file = poses_dir / f"{video_id}.pkl"
    
    try:
        with open(pose_file, 'rb') as f:
            pose = pickle.load(f)
        
        # Check shape
        if len(pose.shape) != 3 or pose.shape[1] != 33 or pose.shape[2] != 3:
            return None, "invalid_shape"
        
        # Check sequence length
        if len(pose) < MIN_FRAMES:
            return None, "too_short"
        if len(pose) > MAX_FRAMES:
            pose = pose[:MAX_FRAMES]  # Truncate
        
        # Check for multi-player
        x_range = pose[:, :, 0].max() - pose[:, :, 0].min()
        if x_range > MULTI_PLAYER_THRESHOLD:
            return None, "multi_player"
        
        return pose, "valid"
    
    except Exception as e:
        return None, f"error_{str(e)[:20]}"

# Filter dataset
print("Filtering poses...")
valid_indices = []
filter_reasons = Counter()

for idx, row in metadata.iterrows():
    _, reason = load_and_filter_pose(row['video_id'])
    filter_reasons[reason] += 1
    
    if reason == "valid":
        valid_indices.append(idx)

# Filter metadata
metadata_filtered = metadata.iloc[valid_indices].reset_index(drop=True)

print(f"\nFiltering results:")
for reason, count in filter_reasons.most_common():
    print(f"  {reason}: {count} ({count/len(metadata)*100:.1f}%)")

print(f"\nFinal dataset: {len(metadata_filtered)} samples")
print(f"Filtered out: {len(metadata) - len(metadata_filtered)} samples")

In [ ]:
# Pose normalization functions
def normalize_pose(pose):
    """
    Normalize pose to make it translation and scale invariant.
    
    Steps:
    1. Center on hip (mid-point of hip joints)
    2. Scale by torso height (hip to nose distance)
    3. Clip outliers
    
    Args:
        pose: (T, 33, 3) array
    
    Returns:
        normalized_pose: (T, 33, 3) array
    """
    # MediaPipe landmark indices
    LEFT_HIP = 23
    RIGHT_HIP = 24
    NOSE = 0
    
    # Calculate hip center (translation reference)
    hip_center = (pose[:, LEFT_HIP, :] + pose[:, RIGHT_HIP, :]) / 2.0
    
    # Center pose on hip
    pose_centered = pose - hip_center[:, np.newaxis, :]
    
    # Calculate torso height (scale reference)
    nose_pos = pose[:, NOSE, :]
    torso_height = np.linalg.norm(nose_pos - hip_center, axis=1)
    torso_height = np.maximum(torso_height, 1e-6)  # Avoid division by zero
    
    # Scale by torso height
    pose_normalized = pose_centered / torso_height[:, np.newaxis, np.newaxis]
    
    # Clip outliers (Z-score > 3)
    pose_normalized = np.clip(pose_normalized, -3, 3)
    
    return pose_normalized.astype(np.float32)

# Test normalization
sample_id = metadata_filtered.iloc[0]['video_id']
sample_pose, _ = load_and_filter_pose(sample_id)

print("Before normalization:")
print(f"  Mean: {sample_pose.mean():.4f}")
print(f"  Std:  {sample_pose.std():.4f}")
print(f"  Min:  {sample_pose.min():.4f}")
print(f"  Max:  {sample_pose.max():.4f}")

sample_pose_norm = normalize_pose(sample_pose)

print("\nAfter normalization:")
print(f"  Mean: {sample_pose_norm.mean():.4f}")
print(f"  Std:  {sample_pose_norm.std():.4f}")
print(f"  Min:  {sample_pose_norm.min():.4f}")
print(f"  Max:  {sample_pose_norm.max():.4f}")

## 4. Dataset and DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Label encoding
SHOT_TYPES = ['Smash', 'Clear', 'Drop', 'Lift', 'Drive']
label_to_idx = {label: idx for idx, label in enumerate(SHOT_TYPES)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

print("Label mapping:")
for label, idx in label_to_idx.items():
    print(f"  {label}: {idx}")

# Split dataset
train_df, test_df = train_test_split(
    metadata_filtered,
    test_size=0.2,
    random_state=42,
    stratify=metadata_filtered['shot_type']
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df['shot_type']
)

print(f"\nDataset splits:")
print(f"  Train: {len(train_df)} samples")
print(f"  Val:   {len(val_df)} samples")
print(f"  Test:  {len(test_df)} samples")

# Check class distribution in splits
print(f"\nTrain distribution:")
print(train_df['shot_type'].value_counts())

In [ ]:
class BadmintonDataset(Dataset):
    """Dataset for badminton pose sequences"""
    
    def __init__(self, df, poses_dir, label_to_idx, normalize=True, max_frames=150):
        self.df = df.reset_index(drop=True)
        self.poses_dir = Path(poses_dir)
        self.label_to_idx = label_to_idx
        self.normalize = normalize
        self.max_frames = max_frames
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load pose
        pose_file = self.poses_dir / f"{row['video_id']}.pkl"
        with open(pose_file, 'rb') as f:
            pose = pickle.load(f)
        
        # Normalize
        if self.normalize:
            pose = normalize_pose(pose)
        
        # Pad or truncate to max_frames
        T, V, C = pose.shape  # T: frames, V: vertices (33), C: coordinates (3)
        
        if T < self.max_frames:
            # Pad with zeros
            pad_length = self.max_frames - T
            pose = np.concatenate([pose, np.zeros((pad_length, V, C))], axis=0)
            actual_frames = T
        else:
            # Truncate
            pose = pose[:self.max_frames]
            actual_frames = self.max_frames
        
        # Convert to tensor: (C, T, V) for ST-GCN
        pose_tensor = torch.from_numpy(pose).permute(2, 0, 1).float()
        
        # Label
        label = self.label_to_idx[row['shot_type']]
        
        return {
            'pose': pose_tensor,
            'label': label,
            'video_id': row['video_id'],
            'actual_frames': actual_frames
        }

# Create datasets
train_dataset = BadmintonDataset(train_df, 'data/poses', label_to_idx, normalize=True)
val_dataset = BadmintonDataset(val_df, 'data/poses', label_to_idx, normalize=True)
test_dataset = BadmintonDataset(test_df, 'data/poses', label_to_idx, normalize=True)

# Create dataloaders
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")

# Test dataloader
sample_batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  Pose shape: {sample_batch['pose'].shape}")  # (B, C, T, V)
print(f"  Label shape: {len(sample_batch['label'])}")

## 5. Model Architectures

### 5.1 ST-GCN (Spatial Temporal Graph Convolutional Network)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class GraphConvolution(nn.Module):
    """Graph convolution layer"""
    
    def __init__(self, in_channels, out_channels, kernel_size=1):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(kernel_size, 1),
            padding=((kernel_size - 1) // 2, 0)
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x, A):
        """Forward pass
        Args:
            x: (N, C, T, V) - input features
            A: (V, V) - adjacency matrix
        Returns:
            x: (N, C, T, V) - output features
        """
        # Apply adjacency matrix (graph convolution)
        x = torch.einsum('nctv,vw->nctw', (x, A))
        
        # Temporal convolution
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        
        return x

class STGCN(nn.Module):
    """ST-GCN for skeleton-based action recognition
    
    Based on: Spatial Temporal Graph Convolutional Networks
    Reference: https://arxiv.org/abs/1801.07455
    """
    
    def __init__(self, num_classes, in_channels=3, num_joints=33, dropout=0.5):
        super().__init__()
        
        # Build adjacency matrix (MediaPipe skeleton connections)
        self.A = self.build_adjacency_matrix(num_joints)
        
        # ST-GCN layers
        self.gcn1 = GraphConvolution(in_channels, 64, kernel_size=9)
        self.gcn2 = GraphConvolution(64, 64, kernel_size=9)
        self.gcn3 = GraphConvolution(64, 64, kernel_size=9)
        self.gcn4 = GraphConvolution(64, 128, kernel_size=9)
        self.gcn5 = GraphConvolution(128, 128, kernel_size=9)
        self.gcn6 = GraphConvolution(128, 128, kernel_size=9)
        self.gcn7 = GraphConvolution(128, 256, kernel_size=9)
        self.gcn8 = GraphConvolution(256, 256, kernel_size=9)
        self.gcn9 = GraphConvolution(256, 256, kernel_size=9)
        
        # Global pooling and classifier
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def build_adjacency_matrix(self, num_joints):
        """Build adjacency matrix for MediaPipe skeleton"""
        # MediaPipe pose connections
        edges = [
            # Face
            (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
            # Arms
            (9, 10), (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21),
            (12, 14), (14, 16), (16, 18), (16, 20), (16, 22),
            # Body
            (11, 23), (12, 24), (23, 24),
            # Legs
            (23, 25), (25, 27), (27, 29), (29, 31),
            (24, 26), (26, 28), (28, 30), (30, 32)
        ]
        
        # Create adjacency matrix
        A = np.zeros((num_joints, num_joints))
        for i, j in edges:
            A[i, j] = 1
            A[j, i] = 1
        
        # Add self-connections
        A = A + np.eye(num_joints)
        
        # Normalize (symmetric normalization)
        D = np.sum(A, axis=1)
        D_inv_sqrt = np.power(D, -0.5)
        D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0
        D_mat_inv_sqrt = np.diag(D_inv_sqrt)
        A_norm = D_mat_inv_sqrt @ A @ D_mat_inv_sqrt
        
        return torch.from_numpy(A_norm).float()
    
    def forward(self, x):
        """Forward pass
        Args:
            x: (N, C, T, V) - pose sequences
        Returns:
            x: (N, num_classes) - class logits
        """
        # Move adjacency matrix to same device as input
        A = self.A.to(x.device)
        
        # ST-GCN blocks
        x = self.gcn1(x, A)
        x = self.gcn2(x, A)
        x = self.gcn3(x, A)
        
        x = self.gcn4(x, A)
        x = self.gcn5(x, A)
        x = self.gcn6(x, A)
        
        x = self.gcn7(x, A)
        x = self.gcn8(x, A)
        x = self.gcn9(x, A)
        
        # Global pooling
        x = self.pool(x)  # (N, C, 1, 1)
        x = x.view(x.size(0), -1)  # (N, C)
        
        # Classifier
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

# Test ST-GCN
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_stgcn = STGCN(num_classes=5, in_channels=3, num_joints=33).to(device)

# Count parameters
total_params = sum(p.numel() for p in model_stgcn.parameters())
trainable_params = sum(p.numel() for p in model_stgcn.parameters() if p.requires_grad)

print(f"ST-GCN Model:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Test forward pass
dummy_input = sample_batch['pose'].to(device)
output = model_stgcn(dummy_input)
print(f"  Output shape: {output.shape}")  # (B, num_classes)

### 5.2 MS-G3D (Multi-Scale Graph 3D)

State-of-the-art graph convolution model with multi-scale aggregation.

In [ ]:
class MultiScaleGraphConv(nn.Module):
    """Multi-scale graph convolution with G3D
    
    Based on: Disentangling and Unifying Graph Convolutions (CVPR 2020)
    Reference: https://arxiv.org/abs/2003.14111
    """
    
    def __init__(self, in_channels, out_channels, num_scales=3, kernel_size=1):
        super().__init__()
        self.num_scales = num_scales
        
        # Multi-scale convolutions
        self.convs = nn.ModuleList([
            nn.Conv2d(
                in_channels,
                out_channels // num_scales,
                kernel_size=(kernel_size, 1),
                padding=((kernel_size - 1) // 2, 0)
            )
            for _ in range(num_scales)
        ])
        
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x, A):
        """Forward pass with multi-scale aggregation"""
        # Multi-scale graph convolution
        out = []
        for i, conv in enumerate(self.convs):
            # Apply different adjacency matrices for different scales
            # Scale 0: local (A), Scale 1: medium (A^2), Scale 2: global (A^3)
            A_scale = torch.matrix_power(A, i + 1) if i > 0 else A
            x_scaled = torch.einsum('nctv,vw->nctw', (x, A_scale))
            x_scaled = conv(x_scaled)
            out.append(x_scaled)
        
        # Concatenate multi-scale features
        x = torch.cat(out, dim=1)
        x = self.bn(x)
        x = self.relu(x)
        
        return x

class MSG3D(nn.Module):
    """MS-G3D: Multi-Scale Graph 3D for action recognition"""
    
    def __init__(self, num_classes, in_channels=3, num_joints=33, dropout=0.5):
        super().__init__()
        
        # Build adjacency matrix
        self.A = self.build_adjacency_matrix(num_joints)
        
        # MS-G3D layers
        self.gcn1 = MultiScaleGraphConv(in_channels, 64, num_scales=3, kernel_size=9)
        self.gcn2 = MultiScaleGraphConv(64, 64, num_scales=3, kernel_size=9)
        self.gcn3 = MultiScaleGraphConv(64, 64, num_scales=3, kernel_size=9)
        self.gcn4 = MultiScaleGraphConv(64, 128, num_scales=3, kernel_size=9)
        self.gcn5 = MultiScaleGraphConv(128, 128, num_scales=3, kernel_size=9)
        self.gcn6 = MultiScaleGraphConv(128, 256, num_scales=3, kernel_size=9)
        
        # Global pooling and classifier
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def build_adjacency_matrix(self, num_joints):
        """Build adjacency matrix (same as ST-GCN)"""
        edges = [
            (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
            (9, 10), (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21),
            (12, 14), (14, 16), (16, 18), (16, 20), (16, 22),
            (11, 23), (12, 24), (23, 24),
            (23, 25), (25, 27), (27, 29), (29, 31),
            (24, 26), (26, 28), (28, 30), (30, 32)
        ]
        
        A = np.zeros((num_joints, num_joints))
        for i, j in edges:
            A[i, j] = 1
            A[j, i] = 1
        A = A + np.eye(num_joints)
        
        D = np.sum(A, axis=1)
        D_inv_sqrt = np.power(D, -0.5)
        D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0
        D_mat_inv_sqrt = np.diag(D_inv_sqrt)
        A_norm = D_mat_inv_sqrt @ A @ D_mat_inv_sqrt
        
        return torch.from_numpy(A_norm).float()
    
    def forward(self, x):
        A = self.A.to(x.device)
        
        x = self.gcn1(x, A)
        x = self.gcn2(x, A)
        x = self.gcn3(x, A)
        x = self.gcn4(x, A)
        x = self.gcn5(x, A)
        x = self.gcn6(x, A)
        
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

# Test MS-G3D
model_msg3d = MSG3D(num_classes=5, in_channels=3, num_joints=33).to(device)

total_params = sum(p.numel() for p in model_msg3d.parameters())
print(f"MS-G3D Model:")
print(f"  Total parameters: {total_params:,}")

output = model_msg3d(dummy_input)
print(f"  Output shape: {output.shape}")

### 5.3 BiLSTM (Bidirectional LSTM)

Temporal baseline for comparison.

In [ ]:
class BiLSTM(nn.Module):
    """Bidirectional LSTM for skeleton-based action recognition"""
    
    def __init__(self, num_classes, in_channels=3, num_joints=33, hidden_size=128, num_layers=2, dropout=0.5):
        super().__init__()
        
        self.num_joints = num_joints
        self.in_channels = in_channels
        
        # Input projection
        self.input_proj = nn.Linear(num_joints * in_channels, hidden_size)
        
        # BiLSTM
        self.lstm = nn.LSTM(
            hidden_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Classifier
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # *2 for bidirectional
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        """Forward pass
        Args:
            x: (N, C, T, V) - pose sequences
        Returns:
            x: (N, num_classes) - class logits
        """
        N, C, T, V = x.shape
        
        # Reshape: (N, T, C*V)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(N, T, C * V)
        
        # Project to hidden size
        x = self.input_proj(x)  # (N, T, hidden_size)
        
        # BiLSTM
        x, (h_n, c_n) = self.lstm(x)  # (N, T, hidden_size*2)
        
        # Take last hidden state
        x = x[:, -1, :]  # (N, hidden_size*2)
        
        # Classifier
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

# Test BiLSTM
model_bilstm = BiLSTM(num_classes=5, in_channels=3, num_joints=33, hidden_size=128).to(device)

total_params = sum(p.numel() for p in model_bilstm.parameters())
print(f"BiLSTM Model:")
print(f"  Total parameters: {total_params:,}")

output = model_bilstm(dummy_input)
print(f"  Output shape: {output.shape}")

### 5.4 Skeleton Transformer

Attention-based model for skeleton sequences.

In [ ]:
class SkeletonTransformer(nn.Module):
    """Transformer for skeleton-based action recognition
    
    Based on recent transformer approaches for skeleton data.
    References:
    - Two-stream GCN-Transformer (2025): https://www.nature.com/articles/s41598-025-87752-8
    - Transformer for skeleton-based action recognition review
    """
    
    def __init__(self, num_classes, in_channels=3, num_joints=33, d_model=256, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        
        self.num_joints = num_joints
        self.in_channels = in_channels
        
        # Input embedding
        self.input_proj = nn.Linear(num_joints * in_channels, d_model)
        
        # Positional encoding
        self.pos_encoding = nn.Parameter(torch.randn(1, 150, d_model))  # max 150 frames
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classifier
        self.fc = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        """Forward pass
        Args:
            x: (N, C, T, V) - pose sequences
        Returns:
            x: (N, num_classes) - class logits
        """
        N, C, T, V = x.shape
        
        # Reshape: (N, T, C*V)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(N, T, C * V)
        
        # Project to d_model
        x = self.input_proj(x)  # (N, T, d_model)
        
        # Add positional encoding
        x = x + self.pos_encoding[:, :T, :]
        
        # Transformer encoding
        x = self.transformer(x)  # (N, T, d_model)
        
        # Global average pooling over time
        x = x.mean(dim=1)  # (N, d_model)
        
        # Classifier
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

# Test Skeleton Transformer
model_transformer = SkeletonTransformer(num_classes=5, in_channels=3, num_joints=33, d_model=256).to(device)

total_params = sum(p.numel() for p in model_transformer.parameters())
print(f"Skeleton Transformer Model:")
print(f"  Total parameters: {total_params:,}")

output = model_transformer(dummy_input)
print(f"  Output shape: {output.shape}")

## 6. Training Setup

In [ ]:
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time

# Training configuration
CONFIG = {
    'num_epochs': 100,
    'learning_rate': 0.001,
    'weight_decay': 0.0001,
    'early_stopping_patience': 15,
    'scheduler': 'cosine',  # 'cosine' or 'plateau'
}

# Class weights for imbalanced dataset
class_counts = train_df['shot_type'].value_counts()
total_samples = len(train_df)
class_weights = torch.tensor([
    total_samples / class_counts[SHOT_TYPES[i]]
    for i in range(len(SHOT_TYPES))
]).float().to(device)

print(f"Class weights:")
for i, shot in enumerate(SHOT_TYPES):
    print(f"  {shot}: {class_weights[i]:.4f}")

# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"\nTraining configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training')
    for batch in pbar:
        poses = batch['pose'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(poses)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        running_loss += loss.item() * poses.size(0)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # Update progress bar
        pbar.set_postfix({'loss': loss.item()})
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    
    return epoch_loss, epoch_acc, epoch_f1

def validate(model, loader, criterion, device):
    """Validate model"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            poses = batch['pose'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(poses)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * poses.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    
    return epoch_loss, epoch_acc, epoch_f1, all_preds, all_labels

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, config, model_name):
    """Complete training loop"""
    best_val_acc = 0.0
    best_epoch = 0
    patience_counter = 0
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'train_f1': [],
        'val_loss': [],
        'val_acc': [],
        'val_f1': [],
    }
    
    print(f"\nTraining {model_name}...")
    print("=" * 80)
    
    start_time = time.time()
    
    for epoch in range(config['num_epochs']):
        print(f"\nEpoch {epoch+1}/{config['num_epochs']}")
        
        # Train
        train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_loss, val_acc, val_f1, _, _ = validate(model, val_loader, criterion, device)
        
        # Update learning rate
        if config['scheduler'] == 'cosine':
            scheduler.step()
        elif config['scheduler'] == 'plateau':
            scheduler.step(val_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        
        # Print metrics
        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}")
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
        print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), f'{model_name}_best.pth')
            print(f"✓ Saved best model (Val Acc: {best_val_acc:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config['early_stopping_patience']:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break
    
    training_time = time.time() - start_time
    
    print(f"\nTraining complete!")
    print(f"Best Val Acc: {best_val_acc:.4f} at epoch {best_epoch}")
    print(f"Training time: {training_time/60:.2f} minutes")
    
    # Load best model
    model.load_state_dict(torch.load(f'{model_name}_best.pth'))
    
    return model, history

print("Training functions defined")

## 7. Train Models

### 7.1 Train ST-GCN

In [ ]:
# Initialize ST-GCN
model_stgcn = STGCN(num_classes=5, in_channels=3, num_joints=33, dropout=0.5).to(device)

# Optimizer and scheduler
optimizer_stgcn = Adam(model_stgcn.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler_stgcn = CosineAnnealingLR(optimizer_stgcn, T_max=CONFIG['num_epochs'])

# Train
model_stgcn, history_stgcn = train_model(
    model_stgcn,
    train_loader,
    val_loader,
    criterion,
    optimizer_stgcn,
    scheduler_stgcn,
    CONFIG,
    'STGCN'
)

### 7.2 Train MS-G3D

In [ ]:
# Initialize MS-G3D
model_msg3d = MSG3D(num_classes=5, in_channels=3, num_joints=33, dropout=0.5).to(device)

# Optimizer and scheduler
optimizer_msg3d = Adam(model_msg3d.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler_msg3d = CosineAnnealingLR(optimizer_msg3d, T_max=CONFIG['num_epochs'])

# Train
model_msg3d, history_msg3d = train_model(
    model_msg3d,
    train_loader,
    val_loader,
    criterion,
    optimizer_msg3d,
    scheduler_msg3d,
    CONFIG,
    'MSG3D'
)

### 7.3 Train BiLSTM

In [ ]:
# Initialize BiLSTM
model_bilstm = BiLSTM(num_classes=5, in_channels=3, num_joints=33, hidden_size=128, dropout=0.5).to(device)

# Optimizer and scheduler
optimizer_bilstm = Adam(model_bilstm.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler_bilstm = CosineAnnealingLR(optimizer_bilstm, T_max=CONFIG['num_epochs'])

# Train
model_bilstm, history_bilstm = train_model(
    model_bilstm,
    train_loader,
    val_loader,
    criterion,
    optimizer_bilstm,
    scheduler_bilstm,
    CONFIG,
    'BiLSTM'
)

### 7.4 Train Skeleton Transformer

In [ ]:
# Initialize Skeleton Transformer
model_transformer = SkeletonTransformer(num_classes=5, in_channels=3, num_joints=33, d_model=256, dropout=0.1).to(device)

# Optimizer and scheduler (AdamW for transformer)
optimizer_transformer = AdamW(model_transformer.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler_transformer = CosineAnnealingLR(optimizer_transformer, T_max=CONFIG['num_epochs'])

# Train
model_transformer, history_transformer = train_model(
    model_transformer,
    train_loader,
    val_loader,
    criterion,
    optimizer_transformer,
    scheduler_transformer,
    CONFIG,
    'Transformer'
)

## 8. Evaluation and Results

In [ ]:
# Evaluate all models on test set
def evaluate_model(model, test_loader, model_name):
    """Comprehensive model evaluation"""
    print(f"\nEvaluating {model_name}...")
    print("=" * 80)
    
    model.eval()
    all_preds = []
    all_labels = []
    all_video_ids = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Testing'):
            poses = batch['pose'].to(device)
            labels = batch['label']
            video_ids = batch['video_id']
            
            outputs = model(poses)
            preds = torch.argmax(outputs, dim=1).cpu()
            
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())
            all_video_ids.extend(video_ids)
    
    # Metrics
    test_acc = accuracy_score(all_labels, all_preds)
    test_f1_macro = f1_score(all_labels, all_preds, average='macro')
    test_f1_weighted = f1_score(all_labels, all_preds, average='weighted')
    
    print(f"\nTest Accuracy: {test_acc:.4f}")
    print(f"Test F1 (Macro): {test_f1_macro:.4f}")
    print(f"Test F1 (Weighted): {test_f1_weighted:.4f}")
    
    # Classification report
    print(f"\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=SHOT_TYPES))
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    return {
        'accuracy': test_acc,
        'f1_macro': test_f1_macro,
        'f1_weighted': test_f1_weighted,
        'predictions': all_preds,
        'labels': all_labels,
        'video_ids': all_video_ids,
        'confusion_matrix': cm
    }

# Evaluate all models
results = {
    'ST-GCN': evaluate_model(model_stgcn, test_loader, 'ST-GCN'),
    'MS-G3D': evaluate_model(model_msg3d, test_loader, 'MS-G3D'),
    'BiLSTM': evaluate_model(model_bilstm, test_loader, 'BiLSTM'),
    'Transformer': evaluate_model(model_transformer, test_loader, 'Transformer'),
}

In [ ]:
# Compare models
print("\n" + "=" * 80)
print("MODEL COMPARISON")
print("=" * 80)

comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Test Accuracy': results[name]['accuracy'],
        'F1 (Macro)': results[name]['f1_macro'],
        'F1 (Weighted)': results[name]['f1_weighted'],
    }
    for name in results.keys()
])

comparison_df = comparison_df.sort_values('Test Accuracy', ascending=False)
print(comparison_df.to_string(index=False))

# Best model
best_model_name = comparison_df.iloc[0]['Model']
best_model_acc = comparison_df.iloc[0]['Test Accuracy']
print(f"\n✓ Best Model: {best_model_name} (Accuracy: {best_model_acc:.4f})")

In [ ]:
# Visualization: Confusion Matrices
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, (name, result) in enumerate(results.items()):
    cm = result['confusion_matrix']
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    sns.heatmap(
        cm_normalized,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=SHOT_TYPES,
        yticklabels=SHOT_TYPES,
        ax=axes[idx]
    )
    axes[idx].set_title(f'{name} - Confusion Matrix\nAccuracy: {result["accuracy"]:.4f}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved confusion matrices to confusion_matrices.png")

In [ ]:
# Visualization: Training History
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

histories = {
    'ST-GCN': history_stgcn,
    'MS-G3D': history_msg3d,
    'BiLSTM': history_bilstm,
    'Transformer': history_transformer,
}

# Loss
for name, history in histories.items():
    axes[0, 0].plot(history['train_loss'], label=f'{name} Train')
    axes[0, 0].plot(history['val_loss'], label=f'{name} Val', linestyle='--')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training and Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Accuracy
for name, history in histories.items():
    axes[0, 1].plot(history['train_acc'], label=f'{name} Train')
    axes[0, 1].plot(history['val_acc'], label=f'{name} Val', linestyle='--')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Training and Validation Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True)

# F1 Score
for name, history in histories.items():
    axes[1, 0].plot(history['val_f1'], label=name)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('F1 Score (Macro)')
axes[1, 0].set_title('Validation F1 Score')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Model comparison
model_names = list(results.keys())
accuracies = [results[name]['accuracy'] for name in model_names]
f1_scores = [results[name]['f1_macro'] for name in model_names]

x = np.arange(len(model_names))
width = 0.35

axes[1, 1].bar(x - width/2, accuracies, width, label='Accuracy')
axes[1, 1].bar(x + width/2, f1_scores, width, label='F1 (Macro)')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_title('Test Performance Comparison')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(model_names)
axes[1, 1].legend()
axes[1, 1].grid(True, axis='y')

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved training history to training_history.png")

## 9. Save Results and Models

In [ ]:
import json
from datetime import datetime

# Create results directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = f"results_{timestamp}"
!mkdir -p {results_dir}

# Save models
print("Saving models...")
torch.save(model_stgcn.state_dict(), f"{results_dir}/stgcn_final.pth")
torch.save(model_msg3d.state_dict(), f"{results_dir}/msg3d_final.pth")
torch.save(model_bilstm.state_dict(), f"{results_dir}/bilstm_final.pth")
torch.save(model_transformer.state_dict(), f"{results_dir}/transformer_final.pth")
print("✓ Models saved")

# Save results summary
print("\nSaving results summary...")
summary = {
    'timestamp': timestamp,
    'dataset': {
        'total_samples': len(metadata_filtered),
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'test_samples': len(test_df),
        'num_classes': len(SHOT_TYPES),
        'shot_types': SHOT_TYPES,
    },
    'config': CONFIG,
    'results': {
        name: {
            'accuracy': float(result['accuracy']),
            'f1_macro': float(result['f1_macro']),
            'f1_weighted': float(result['f1_weighted']),
        }
        for name, result in results.items()
    },
    'best_model': {
        'name': best_model_name,
        'accuracy': float(best_model_acc),
    }
}

with open(f"{results_dir}/results_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)
print("✓ Results summary saved")

# Save detailed results
print("\nSaving detailed results...")
comparison_df.to_csv(f"{results_dir}/model_comparison.csv", index=False)

for name, result in results.items():
    # Save predictions
    pred_df = pd.DataFrame({
        'video_id': result['video_ids'],
        'true_label': [SHOT_TYPES[i] for i in result['labels']],
        'pred_label': [SHOT_TYPES[i] for i in result['predictions']],
        'correct': [result['labels'][i] == result['predictions'][i] for i in range(len(result['labels']))]
    })
    pred_df.to_csv(f"{results_dir}/{name.lower().replace('-', '_')}_predictions.csv", index=False)
    
    # Save confusion matrix
    cm_df = pd.DataFrame(
        result['confusion_matrix'],
        index=[f'True_{shot}' for shot in SHOT_TYPES],
        columns=[f'Pred_{shot}' for shot in SHOT_TYPES]
    )
    cm_df.to_csv(f"{results_dir}/{name.lower().replace('-', '_')}_confusion_matrix.csv")

print("✓ Detailed results saved")

# Copy visualizations
!cp confusion_matrices.png {results_dir}/
!cp training_history.png {results_dir}/

print(f"\n✓ All results saved to: {results_dir}/")
print(f"\nUpload to GCS:")
print(f"  !gsutil -m cp -r {results_dir} gs://iti123storage/outputs/")

## 10. Summary and Next Steps

In [ ]:
print("=" * 80)
print("TRAINING COMPLETE - SUMMARY")
print("=" * 80)
print(f"\nDataset:")
print(f"  Total samples: {len(metadata_filtered)}")
print(f"  Train/Val/Test: {len(train_df)}/{len(val_df)}/{len(test_df)}")
print(f"  Shot types: {', '.join(SHOT_TYPES)}")

print(f"\nModel Performance:")
print(comparison_df.to_string(index=False))

print(f"\n✓ Best Model: {best_model_name}")
print(f"  Test Accuracy: {best_model_acc:.4f}")
print(f"  F1 (Macro): {results[best_model_name]['f1_macro']:.4f}")

print(f"\nResults saved to: {results_dir}/")
print(f"\nNext Steps:")
print(f"  1. Upload results to GCS:")
print(f"     !gsutil -m cp -r {results_dir} gs://iti123storage/outputs/")
print(f"  2. Analyze per-class performance")
print(f"  3. Try data augmentation")
print(f"  4. Ensemble models")
print(f"  5. Deploy best model")
print("=" * 80)

---

## Research References

### Badminton-Specific Research (2024-2025)
1. [Deep learning-based badminton action recognition and quality assessment](https://journals.sagepub.com/doi/10.1177/1088467X251353444) - SlowFast + Siamese Network
2. [Strategy analysis of badminton players using deep learning from IMU and UWB wearables](https://www.sciencedirect.com/science/article/abs/pii/S2542660524002014) - 2D-CNN + LSTM
3. [The analysis of motion recognition model for badminton player movements](https://www.nature.com/articles/s41598-025-02771-9) - VGG16-BiLSTM-CBAM
4. [BST: Badminton Stroke-type Transformer](https://arxiv.org/html/2502.21085) - Transformer for racket sports

### Graph Convolutional Networks
5. [ST-GCN: Spatial Temporal Graph Convolutional Networks](https://arxiv.org/abs/1801.07455) - Original ST-GCN paper
6. [MS-G3D: Disentangling and Unifying Graph Convolutions](https://arxiv.org/abs/2003.14111) - CVPR 2020, multi-scale GCN
7. [Two-stream spatio-temporal GCN-transformer networks](https://www.nature.com/articles/s41598-025-87752-8) - Recent GCN + Transformer

### Transformer-Based Methods
8. [Transformer for Skeleton-based action recognition review](https://www.sciencedirect.com/science/article/abs/pii/S0925231223002217)

---

**Notebook Version:** 1.0  
**Last Updated:** 2026-02-03  
**Author:** Phase 1.5 ROI Extraction Pipeline